In [1]:
import nest_asyncio
nest_asyncio.apply()

from IPython.display import Markdown, display

In [2]:
from typing import List
from dotenv import dotenv_values

from llama_index.core import (SimpleDirectoryReader, 
                              Settings,
                              VectorStoreIndex)

from llama_index.core.ingestion import IngestionPipeline

from llama_index.core.node_parser import SentenceSplitter

from llama_index.core.schema import (BaseNode,
                                     Document,
                                     MetadataMode)

from llama_index.vector_stores.qdrant import QdrantVectorStore

from llama_parse import LlamaParse

from qdrant_client import QdrantClient

from utils import add_metadata_to_documents

In [3]:
from nvidia_nims import (embedding_model,
                         llm,
                         rerank_model)

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
/workspaces/Marketing_Research_Agent/nvidia-ai/lib/python3.12/site-packages/llama_index/llms/nvidia/base.py:178: UserWarning: Unable to determine validity of meta/llama-3.2-1b-instruct
  warnings.warn(f"Unable to determine validity of {model_name}")
/workspaces/Marketing_Research_Agent/nvidia-ai/lib/python3.12/site-packages/llama_index/postprocessor/nvidia_rerank/base.py:232: UserWarning: Unable to determine validity of nvidia/llama-3.2-nv-embedqa-1b-v1
  warnings.warn(f"Unable to determine validity of {model_name}")


In [4]:
config = dotenv_values(".env")

In [5]:
def extract(pdf_document: str = ["./sample_data/ozempic.pdf"], language: str = "en", target_pages: str = None):

    parsing_instructions = """
    The provided document is a thin piece of folded paper that is part of every drug prescription box. 
    Usually the text is in VERY small print and typically provides information about dosages, side effects, storage instructions and much more. 
    Try to extract the key information so that it is easy to understand.
    """

    pdf_parser = LlamaParse(
    api_key=config["LLAMACLOUD_API_KEY"],
    result_type="text",  # markdown doesn't work with fast_mode to True
    parsing_instruction=parsing_instructions,
    num_workers=7,
    check_interval=2,
    max_timeout=2000,
    verbose=True,
    show_progress=True,
    language=language,
    invalidate_cache=False,
    do_not_cache=False,
    fast_mode=True, # fast_mode=True doesn't work with result_type="markdown"
    ignore_errors=True,
    split_by_page=True,
    disable_ocr=True,
    target_pages=target_pages  # for testing purposes use target_pages="0,80" to only parse the first and last page 
    )

    file_extractor = {".pdf": pdf_parser}

    documents = SimpleDirectoryReader(input_files=pdf_document, 
                                      file_extractor=file_extractor,
                                      filename_as_id=True,
                                      required_exts=[".pdf"],
                                      num_files_limit=1).load_data()
    
    return documents

In [6]:
documents = extract()

Started parsing the file under job_id 8cdc4bf9-a609-46a5-8238-e72c79bc6709


In [7]:
print(len(documents))

81


In [8]:
documents = add_metadata_to_documents(documents)

In [9]:
documents[10].metadata

{'file_path': 'sample_data/ozempic.pdf',
 'file_name': 'ozempic.pdf',
 'file_type': 'application/pdf',
 'file_size': 2072988,
 'creation_date': '2024-10-25',
 'last_modified_date': '2024-10-25',
 'total_pages_in_original_pdf': 81,
 'size_of_original_pdf(MB)': '1.98 MB'}

In [10]:
def transform_documents(documents: List[Document]) -> List[Document]:
    transformed_documents = []
    for document in documents:
        transformed_documents.append(
            Document(
                text=document.text,
                metadata=document.metadata,
                excluded_llm_metadata_keys=["file_name", "file_path", "file_type", "file_size", "creation_date", "last_modified_date", "total_pages_in_original_pdf", "size_of_original_pdf(MB)"],
                excluded_embed_metadata_keys = ["file_path", "file_type", "file_size", "creation_date", "last_modified_date", "total_pages_in_original_pdf", "size_of_original_pdf(MB)"],
                metadata_seperator="::",
                metadata_template="{key}=>{value}",
                text_template="Metadata: {metadata_str}\n-----\nContent: {content}",
            )
        )
    return transformed_documents

In [11]:
documents = transform_documents(documents)

In [12]:
documents[1]

Document(id_='203ae839-38da-4d61-8faa-35b429061eea', embedding=None, metadata={'file_path': 'sample_data/ozempic.pdf', 'file_name': 'ozempic.pdf', 'file_type': 'application/pdf', 'file_size': 2072988, 'creation_date': '2024-10-25', 'last_modified_date': '2024-10-25', 'total_pages_in_original_pdf': 81, 'size_of_original_pdf(MB)': '1.98 MB'}, excluded_embed_metadata_keys=['file_path', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'total_pages_in_original_pdf', 'size_of_original_pdf(MB)'], excluded_llm_metadata_keys=['file_name', 'file_path', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'total_pages_in_original_pdf', 'size_of_original_pdf(MB)'], relationships={}, text='  •   Personal or family history of MTC or in patients with MEN 2 (4).\n  •   Serious hypersensitivity reaction to semaglutide or any of the excipients in OZEMPIC (4).\n\n                                          WARNINGS AND PRECAUTIONS\n\n  •   Pancreatitis: Has been reported in clin

In [13]:
print(
    "The LLM sees this: \n",
    documents[1].get_content(metadata_mode=MetadataMode.LLM),
)
print(
    "The Embedding model sees this: \n",
    documents[1].get_content(metadata_mode=MetadataMode.EMBED),
)

The LLM sees this: 
   •   Personal or family history of MTC or in patients with MEN 2 (4).
  •   Serious hypersensitivity reaction to semaglutide or any of the excipients in OZEMPIC (4).

                                          WARNINGS AND PRECAUTIONS

  •   Pancreatitis: Has been reported in clinical trials. Discontinue promptly if pancreatitis is suspected. Do
      not restart if pancreatitis is confirmed (5.2).
  •   Diabetic Retinopathy Complications: Has been reported in a clinical trial. Patients with a history of
      diabetic retinopathy should be monitored (5.3).
  •   Never share an OZEMPIC pen between patients, even if the needle is changed (5.4).
  •   Hypoglycemia: Concomitant use with an insulin secretagogue or insulin may increase the risk of
      hypoglycemia, including severe hypoglycemia. Reducing dose of insulin secretagogue or insulin may
      be necessary (5.5).
  •   Acute Kidney Injury: Monitor renal function in patients with renal impairment reporting se

In [14]:
from llama_index.embeddings.nvidia import NVIDIAEmbedding
Settings.text_splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=50)
Settings.embed_model = NVIDIAEmbedding(model="nvidia/llama-3.2-nv-embedqa-1b-v1", truncate="END", api_key=config["EMBED_MODEL_API_KEY"])

/workspaces/Marketing_Research_Agent/nvidia-ai/lib/python3.12/site-packages/llama_index/embeddings/nvidia/base.py:205: UserWarning: Unable to determine validity of nvidia/llama-3.2-nv-embedqa-1b-v1
  warnings.warn(f"Unable to determine validity of {model_name}")


In [15]:
index = VectorStoreIndex.from_documents(documents)

AuthenticationError: Error code: 401 - {'status': 401, 'title': 'Unauthorized', 'detail': 'invalid response from UAM'}

In [20]:
Settings.llm = llm

query_engine = index.as_query_engine()

In [ ]:
query_engine.query("ozempic")

In [ ]:
response = query_engine.query("What are the side effects of Ozempic?")
print(response)

In [26]:
def load(documents: List[Document]):
    text_splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=20)
    qdrant_client = QdrantClient(url=config["QDRANT_ENDPOINT"], 
                             api_key=config["QDRANT_API_KEY"])
    
    vector_store = QdrantVectorStore(client=qdrant_client, collection_name="pillpal_documents")
    
    pipeline = IngestionPipeline(
        transformations=[text_splitter, embedding_model],
        vector_store=vector_store,
    )
    nodes = pipeline.run(documents=documents)

    index = VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=embedding_model)

    return nodes, index

In [ ]:
nodes, index = load(documents)